# TP2 — Classification : qui a survécu au Titanic ? (Énoncé)

**Durée : 2 h** · Cours associé : [TP2_Cours.md](TP2_Cours.md) · Pense-bête : [Pense_bete_Pandas.md](../Pense_bete_Pandas.md)

## Contexte

Le 15 avril 1912, le Titanic sombre : 1 502 des 2 224 passagers et membres d'équipage périssent. Le fichier `data/titanic.csv` décrit 891 passagers (classe du billet, sexe, âge, tarif, port d'embarquement…) et indique s'ils ont survécu (`Survived = 1`).

**Objectif** : construire un modèle qui prédit la survie d'un passager à partir de ses caractéristiques, et évaluer honnêtement ses performances.

## Déroulé

| Partie | Contenu | Durée indicative |
|---|---|---|
| 1 | Exploration des données | 20 min |
| 2 | Découpage train / test et baseline | 10 min |
| 3 | Pipeline de préparation et régression logistique | 25 min |
| 4 | Métriques d'évaluation | 20 min |
| 5 | k-NN, arbre de décision et sur-apprentissage | 25 min |
| 6 | Validation croisée et choix du modèle | 20 min |

Les cellules marquées `# TODO` sont à compléter. Les cellules ✍️ attendent une réponse rédigée.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)

plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42

## Partie 1 — Exploration des données (20 min)

| Colonne | Signification |
|---|---|
| `Survived` | 1 = survivant, 0 = décédé (**cible**) |
| `Pclass` | classe du billet (1 = première, 3 = troisième) |
| `Sex`, `Age` | sexe et âge |
| `SibSp` | nombre de frères, sœurs ou conjoints à bord |
| `Parch` | nombre de parents ou enfants à bord |
| `Fare` | prix du billet |
| `Embarked` | port d'embarquement (C = Cherbourg, Q = Queenstown, S = Southampton) |
| `Name`, `Ticket`, `Cabin` | nom, numéro de billet, numéro de cabine |

**1.1** Chargez les données dans `df`, affichez les premières lignes et le nombre de valeurs manquantes par colonne.

In [ ]:
# TODO : chargement et valeurs manquantes


**1.2** Quel est le taux de survie global ? Calculez ensuite le taux de survie par sexe, par classe, puis par sexe **et** classe (tableau croisé).

In [ ]:
# TODO : taux de survie global, par sexe, par classe, par sexe et classe


**1.3** Comparez la distribution de l'âge des survivants et des victimes (deux histogrammes superposés). Créez ensuite une colonne `Enfant` (âge < 12 ans) et calculez le taux de survie des enfants.

In [ ]:
# TODO : âge et survie


✍️ **Votre réponse :**

…

**1.4** Quelles colonnes vous semblent inutilisables telles quelles pour un modèle ? Justifiez.

✍️ **Votre réponse :**

…

## Partie 2 — Découpage train / test et baseline (10 min)

**2.1** Construisez `X` (les 7 variables retenues) et `y` (`Survived`), puis découpez en 80 % entraînement / 20 % test, de façon **stratifiée**, avec `random_state=RANDOM_STATE`. Vérifiez que le taux de survie est le même dans les deux jeux.

In [ ]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
# TODO : X, y et découpage stratifié

**2.2** Entraînez un `DummyClassifier(strategy="most_frequent")` et calculez son exactitude sur le jeu de test. Que prédit-il ? Pourquoi ce score est-il important ?

In [ ]:
# TODO : baseline


✍️ **Votre réponse :**

…

## Partie 3 — Pipeline de préparation et régression logistique (25 min)

**3.1** Construisez un `ColumnTransformer` nommé `preprocess` qui :

- pour les colonnes numériques `Age`, `Fare`, `SibSp`, `Parch` : impute par la **médiane** puis **standardise** ;
- pour les colonnes catégorielles `Sex`, `Embarked`, `Pclass` : impute par la **modalité la plus fréquente** puis applique un **one-hot encoding** (`handle_unknown="ignore"`).

*Remarque : on traite `Pclass` comme une catégorie. Pourquoi est-ce discutable ? Pourquoi est-ce raisonnable ?*

In [ ]:
num_cols = ["Age", "Fare", "SibSp", "Parch"]
cat_cols = ["Sex", "Embarked", "Pclass"]
# TODO : ColumnTransformer

**3.2** Pour comprendre ce que fait `preprocess`, appliquez-le au jeu d'entraînement (`fit_transform`) et affichez le résultat sous forme de DataFrame avec les noms de colonnes produits (`get_feature_names_out()`).

In [ ]:
# TODO : observer la sortie du préprocesseur


**3.3** Créez un `Pipeline` `logreg` qui enchaîne `preprocess` et une `LogisticRegression(max_iter=1000)`. Entraînez-le et affichez l'exactitude sur le jeu d'entraînement et sur le jeu de test.

In [ ]:
# TODO : pipeline régression logistique


**3.4** Affichez les coefficients de la régression logistique en face des noms de variables, triés. Lesquels augmentent la probabilité de survie ? Est-ce cohérent avec la partie 1 ?

*Indice : `logreg.named_steps["clf"].coef_[0]` et `logreg.named_steps["prep"].get_feature_names_out()`.*

In [ ]:
# TODO : coefficients


✍️ **Votre réponse :**

…

## Partie 4 — Métriques d'évaluation (20 min)

**4.1** Affichez la matrice de confusion de `logreg` sur le jeu de test (`ConfusionMatrixDisplay.from_estimator`) et le `classification_report`. Combien de survivants le modèle a-t-il « ratés » ?

In [ ]:
# TODO : matrice de confusion et rapport


**4.2** Recalculez **à la main** (à partir des quatre cases de la matrice) la précision et le rappel de la classe « survivant », puis vérifiez avec `precision_score` et `recall_score`.

In [ ]:
# TODO : précision et rappel à la main


**4.3** Tracez la courbe ROC et calculez l'AUC. Puis, en utilisant `predict_proba`, calculez précision et rappel pour des seuils de décision de 0,3, 0,5 et 0,7. Que constatez-vous ?

In [ ]:
# TODO : ROC, AUC et effet du seuil


✍️ **Votre réponse :**

…

## Partie 5 — k-NN, arbre de décision et sur-apprentissage (25 min)

**5.1** Créez un pipeline `knn` (même `preprocess` + `KNeighborsClassifier(n_neighbors=5)`) et un pipeline `arbre` (`preprocess` + `DecisionTreeClassifier(random_state=RANDOM_STATE)` sans limite de profondeur). Pour chacun, affichez l'exactitude en entraînement et en test.

*Astuce : réutiliser le même objet `preprocess` dans plusieurs pipelines pose problème (il serait partagé). Utilisez `sklearn.base.clone(preprocess)`.*

In [ ]:
from sklearn.base import clone
# TODO : k-NN et arbre non contraint

**5.2** Faites varier la profondeur maximale de l'arbre (`max_depth` de 1 à 15) et tracez l'exactitude en entraînement et en test en fonction de la profondeur. Identifiez les zones de sous-apprentissage et de sur-apprentissage.

*Astuce : `arbre.set_params(clf__max_depth=d)` modifie un hyperparamètre d'une étape du pipeline.*

In [ ]:
# TODO : courbe de complexité de l'arbre


✍️ **Votre réponse :**

…

**5.3** Entraînez un arbre de profondeur 3 et affichez-le avec `plot_tree` (paramètres utiles : `feature_names`, `class_names`, `filled=True`). Quelles sont les premières questions posées par l'arbre ?

In [ ]:
# TODO : visualiser un arbre peu profond


✍️ **Votre réponse :**

…

## Partie 6 — Validation croisée et choix du modèle (20 min)

**6.1** Évaluez les trois pipelines (`logreg`, `knn` et `arbre` avec `max_depth=None`) par **validation croisée à 5 plis sur le jeu d'entraînement** (`cross_val_score`). Affichez la moyenne et l'écart-type pour chacun.

In [ ]:
# TODO : validation croisée des trois modèles


**6.2** Utilisez `GridSearchCV` pour régler simultanément `max_depth` (2 à 10) et `min_samples_leaf` (1, 5, 10, 20) de l'arbre, en validation croisée à 5 plis. Affichez les meilleurs hyperparamètres et le score de validation croisée correspondant.

In [ ]:
# TODO : recherche d'hyperparamètres pour l'arbre


**6.3** Faites de même pour le k-NN avec `n_neighbors` de 1 à 30.

In [ ]:
# TODO : recherche d'hyperparamètres pour le k-NN


**6.4** Choisissez **un** modèle final à partir des scores de validation croisée, puis évaluez-le **une seule fois** sur le jeu de test (exactitude, F1 et AUC). Concluez.

In [ ]:
# TODO : évaluation finale


✍️ **Votre réponse :**

…

## Pour aller plus loin (bonus)

**Ingénierie de variables** : le nom contient un titre (*Mr*, *Mrs*, *Miss*, *Master*…) qui combine sexe, âge et statut social. Extrayez-le dans une colonne `Titre` (regroupez les titres rares dans « Autre »), ajoutez une variable `TailleFamille = SibSp + Parch + 1`, puis mesurez le gain de la régression logistique en validation croisée.

*Indice : `df["Name"].str.extract(r",\s*([^\.]+)\.")`.*

In [ ]:
# Bonus : titre et taille de la famille
